# Annualized volatility predictions

Author: Pete King

In our analysis, we measured performance of three different types of models to predict realized volatility for an asset: a Ridge regressor, a Gradient Boosting Regressor (GBR), and a Long Short-Term Memory (LSTM) deep-learning model.  For each model type, we tested performance using three different sets of input features: a 'limited' set relying primarily on historical (realized) volatility, a 'full' set including 87 engineered features, and a 'PCA' set of 13 extracted features (along with historical return and historical volatility).

We conducted a preliminary analysis to asses the best-performing combination of model type and feature set according to a set of criteria, testing the performance of those models using a validation set.  From these preliminary tests, we selected the LSTM models, since those using the 'limited' or 'PCA' set of features displayed superior performance according to our evaluation criteria.

Then, for the chosen model type, we conducted 30 trials.  In each trial, we trained the models on the training set data and then used them to provide a set of weekly predictions (forecast) for **realized** volatility.  We collected predictions for the training set, validation set, and test set, but we only consider performance on the test set in this notebook.

The final component of our "Portfolio Generator" requires a prediction of **annualized** volatility for each ETF.  Here we convert the model's predictions of realized volatility to annualized volatility for each ETF, and compute two values:

 - An average value for annualized volatility over the entire test set (over 30 trials, with 61 weekly predictions in each trial)

 - A median value of annualized volatility for the latest available prediction in the test set (again over 30 trials, but considering only the final prediction for Week 61)

The first figure represents our best estimate of overall typical **long-term** volatility for the asset during the past 61 weeks (i.e., the testing period), while the second figure represents our estimate of **near-term** future volatility -- a prediction for the next 30 days.

Our approach provides two key advantages over simply relying on the Cboe VIX (S&P 500 Volatility Index) as a measure of asset volatility:

 - We provide a sector-based estimate, which as you will see from the data can differ significantly from overall S&P 500 volatility.

 - Our LSTM model with 'limited' input consistently achieves a higher out-of-sample R-squared score for volatility predictions compared to the VIX.


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import altair as alt

DATA_DIR = Path.cwd().parent / 'data'
NUM_TRIALS = 30

In [2]:
with open(DATA_DIR / 'volatility_results.json') as f:
    results = json.load(f)

In [3]:
etfs = list(results['1'].keys())
etfs

['BIL',
 'BND',
 'GLD',
 'HYG',
 'IEF',
 'IWM',
 'LQD',
 'QQQ',
 'SPY',
 'TIP',
 'TLT',
 'XLB',
 'XLE',
 'XLF',
 'XLI',
 'XLK',
 'XLP',
 'XLRE',
 'XLU',
 'XLV',
 'XLY']

In [4]:
# For each ETF, collect all sets of weekly predictions as a DataFrame
num_forecasts_per_trial = len(
    results['1'][etfs[0]]['LSTM']['limited']['test']['y_pred']
)
y_pred = {}
for etf in etfs:
    rows = []
    for i in range(NUM_TRIALS):
        # Add each trial as a row vector
        rows.append(
            results[str(i + 1)][etf]['LSTM']['limited']['test']['y_pred']
        )
    y_pred[etf] = pd.DataFrame(
        rows,
        columns=[f'Week_{i + 1}' for i in range(num_forecasts_per_trial)],
        index=pd.Series([str(i + 1) for i in range(NUM_TRIALS)], name='Trial')
    )

In [5]:
y_pred['SPY']

,Week_1,Week_2,Week_3,Week_4,Week_5,Week_6,Week_7,Week_8,Week_9,Week_10,...,Week_52,Week_53,Week_54,Week_55,Week_56,Week_57,Week_58,Week_59,Week_60,Week_61
Trial,,,,,,,,,,,,,,,,,,,,,
1,0.008724,0.009999,0.008617,0.010262,0.008527,0.008741,0.006798,0.007954,0.011197,0.009482,...,0.008874,0.008807,0.009422,0.010264,0.010816,0.010737,0.009148,0.008347,0.009173,0.008844
2,0.009543,0.011116,0.009215,0.009315,0.010147,0.008970,0.005752,0.008144,0.013302,0.011098,...,0.009887,0.009552,0.009800,0.011293,0.012099,0.011337,0.009606,0.008757,0.009759,0.009754
3,0.008852,0.010756,0.008483,0.009417,0.007761,0.008457,0.006786,0.009231,0.012347,0.008602,...,0.008578,0.007470,0.009323,0.010019,0.010387,0.010528,0.010159,0.008993,0.009023,0.009402
4,0.009581,0.010576,0.009631,0.011395,0.009312,0.009688,0.007859,0.008860,0.011682,0.010099,...,0.009473,0.009496,0.010081,0.010744,0.011009,0.010882,0.010217,0.009789,0.010153,0.009392
5,0.008973,0.010785,0.007728,0.009055,0.009150,0.008550,0.005813,0.007600,0.012482,0.010450,...,0.009122,0.008562,0.009528,0.010870,0.011532,0.010580,0.008882,0.008311,0.009367,0.009172
6,0.009399,0.010765,0.009740,0.010839,0.009232,0.009174,0.007550,0.008519,0.012197,0.010479,...,0.009803,0.009440,0.010101,0.010928,0.011414,0.011194,0.010025,0.009166,0.009799,0.009454
7,0.009710,0.011064,0.008384,0.009550,0.009974,0.009225,0.006295,0.008459,0.012240,0.010510,...,0.009308,0.009643,0.009850,0.011252,0.011780,0.010937,0.009307,0.008873,0.009972,0.009723
8,0.009684,0.011003,0.009494,0.010958,0.009731,0.009601,0.007352,0.008663,0.011913,0.010753,...,0.009967,0.009793,0.009961,0.011139,0.011658,0.011205,0.009910,0.009282,0.009959,0.009658
9,0.009463,0.010799,0.009484,0.011070,0.009635,0.009605,0.007226,0.008520,0.012034,0.010457,...,0.009597,0.009777,0.009975,0.010931,0.011424,0.011032,0.009972,0.009435,0.009848,0.009570


In [6]:
# Get the overall average volatility for each ETF during the test set period
print(f'Median annualized volatility prediction (over {
    NUM_TRIALS} trials on test set)\n---')
avg_annualized_vol = {}
for etf in etfs:
    avg_annualized_vol[etf] = y_pred[etf].mean().mean() * np.sqrt(252) * 100
avg_annualized_vol = pd.Series(
    avg_annualized_vol, name='Average Volatility (test set)'
)
avg_annualized_vol

Median annualized volatility prediction (over 30 trials on test set)
---


BIL      0.455343
BND      5.981263
GLD     17.093881
HYG      4.115452
IEF      6.526064
IWM     24.749133
LQD      7.831116
QQQ     22.536702
SPY     17.454022
TIP      5.498059
TLT     15.862322
XLB     20.572965
XLE     25.802990
XLF     22.216566
XLI     19.248253
XLK     25.005235
XLP     11.739381
XLRE    18.243003
XLU     16.848320
XLV     14.922235
XLY     24.739104
Name: Average Volatility (test set), dtype: float64

In [7]:
# Take an average of <30> trials of the latest predition on the test set
print(f'Most recent annualized volatility prediction (median for {
    NUM_TRIALS} trials)\n---')
recent_annualized_vol = {}
for etf in etfs:
    recent_annualized_vol[etf] = (
        np.median(y_pred[etf].values[:, -1]) * np.sqrt(252) * 100
    )
recent_annualized_vol = pd.Series(
    recent_annualized_vol, name='Recent Volatility (latest prediction)'
)
recent_annualized_vol

Most recent annualized volatility prediction (median for 30 trials)
---


BIL      0.401787
BND      4.718726
GLD     15.323170
HYG      2.878141
IEF      5.371611
IWM     23.034889
LQD      6.649408
QQQ     20.668170
SPY     14.966995
TIP      4.504144
TLT     14.339533
XLB     17.732679
XLE     27.833129
XLF     19.739298
XLI     18.014827
XLK     23.422370
XLP     11.419811
XLRE    16.231646
XLU     17.332471
XLV     14.888880
XLY     22.485580
Name: Recent Volatility (latest prediction), dtype: float64

In [8]:
df = pd.DataFrame([avg_annualized_vol, recent_annualized_vol]).T

In [9]:
df

,Average Volatility (test set),Recent Volatility (latest prediction)
BIL,0.455343,0.401787
BND,5.981263,4.718726
GLD,17.093881,15.323170
HYG,4.115452,2.878141
IEF,6.526064,5.371611
IWM,24.749133,23.034889
LQD,7.831116,6.649408
QQQ,22.536702,20.668170
SPY,17.454022,14.966995
TIP,5.498059,4.504144


In [10]:
df.to_csv(
    DATA_DIR / 'annualized_volatility_predictions.csv',
    index_label='ETF'
)